In [1]:
import sys
sys.path.append('../..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [2]:
df = read_table(
    """
    select distinct(age_group)
    from sc_bronze.dosm_graduates_age
    """
)

df.head()

,age_group
0,35 - 44
1,≥ 45
2,25 - 34
3,≤ 24


In [3]:
order_map = {
    "≤ 24": 1,
    "25 - 34": 2,
    "35 - 44": 3,
    "≥ 45": 4,
}

def order_age_group(x):
    x_str = str(x).strip()
    if x_str in order_map:
        return order_map[x_str]
    if "-" in x_str:
        return int(x_str.split("-")[0].strip())
    if x_str.startswith("≤"):
        return 1
    if x_str.startswith("≥"):
        return 4

df = df.copy()
df['age_group_order'] = df['age_group'].apply(order_age_group)
df = df.sort_values(['age_group_order', 'age_group']).reset_index(drop=True)
df = df.drop(columns=['age_group_order'])
df["age_group_id"] = ["AG" + str(i+1).zfill(3) for i in range(len(df))]
df = df[["age_group_id"] + [c for c in df.columns if c != "age_group_id"]]

In [4]:
write_table(df, "sc_gold", "dim_agegroup")

Table sc_gold.dim_agegroup written successfully.
